# Simulazione di Gestione Crisi Idrica
## Integrazione Dominio Idrico e Sensoristico

Questo progetto unisce:
- **WNTR**: per la simulazione della rete idrica (Water Network Tool for Resilience)
- **Simulazione LoRaWAN integrata**: per la simulazione della rete di sensori

**Obiettivo**: Gestire una crisi idrica attraverso serbatoi intelligenti controllati da un agente centrale.

## 1. Import delle Librerie

In [ ]:
import sys
import os
import random
import math
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import OrderedDict

# Importa WNTR (già installato via pip)
try:
    import wntr
    print(f"✅ WNTR importato con successo (versione: {wntr.__version__})")
except ImportError as e:
    print(f"❌ Errore nell'import di WNTR: {e}")
    print("Installa WNTR con: pip install wntr")
    raise

print("\n✅ Ambiente configurato correttamente!")

## 2. Simulazione Rete LoRaWAN

In [ ]:
@dataclass
class Packet:
    """Rappresenta un pacchetto LoRaWAN."""
    node_id: str
    data: float
    timestamp: float
    sf: int = 7


class Gateway:
    """Gateway LoRaWAN che riceve i pacchetti dai sensori."""

    def __init__(self):
        self.received_packets: List[Packet] = []
        self.active_transmissions: List[Packet] = []
        self.stat_totale_inviati: int = 0
        self.stat_totale_persi: int = 0
        self.stat_totale_ricevuti: int = 0

    def receive_uplink(self, packet: Packet, time_on_air: float, current_time: float) -> bool:
        """Simula la ricezione di un uplink con possibile collisione."""
        self.stat_totale_inviati += 1
        
        # Simula collisione: se ci sono già trasmissioni attive nello stesso momento
        collision = len(self.active_transmissions) > 0
        
        if not collision:
            self.active_transmissions.append(packet)
            # Il pacchetto viene ricevuto con successo dopo il time_on_air
            self.received_packets.append(packet)
            self.stat_totale_ricevuti += 1
            self.active_transmissions.remove(packet)
            return True
        else:
            # Collisione: pacchetto perso
            self.stat_totale_persi += 1
            return False

    def get_packet_loss_rate(self) -> float:
        """Calcola la percentuale di pacchetti persi."""
        if self.stat_totale_inviati == 0:
            return 0.0
        return (self.stat_totale_persi / self.stat_totale_inviati) * 100.0

    def get_and_clear_buffer(self) -> List[Packet]:
        """Restituisce e pulisce il buffer dei pacchetti ricevuti."""
        data = self.received_packets.copy()
        self.received_packets.clear()
        return data


class SensorNode:
    """Nodo sensore che misura pressione e invia dati via LoRaWAN."""

    def __init__(self, node_id: str, gateway: Gateway,
                 sf: int = 7, tx_interval: int = 3600):
        self.node_id = node_id
        self.gateway = gateway
        self.sf = sf
        self.tx_interval = tx_interval
        self.current_pressure: float = 0.0
        self.current_tank_level: float = 0.0
        self.next_tx_time: float = random.uniform(0, min(tx_interval, 100))
        # Time on air dipende dallo Spreading Factor (in secondi simulati)
        self.time_on_air = (2 ** self.sf) / 125000.0 * 20 * 1000 / 1000.0

    def update_sensor_data(self, pressure: float, tank_level: float = 0.0):
        """Aggiorna i dati del sensore."""
        self.current_pressure = pressure
        self.current_tank_level = tank_level

    def update_tx_interval(self, new_interval: int):
        """Aggiorna l'intervallo di trasmissione."""
        self.tx_interval = new_interval

    def try_transmit(self, current_time: float) -> bool:
        """Tenta di trasmettere se è arrivato il momento."""
        if current_time >= self.next_tx_time:
            packet = Packet(
                node_id=self.node_id,
                data=self.current_pressure,
                timestamp=current_time,
                sf=self.sf
            )
            success = self.gateway.receive_uplink(packet, self.time_on_air, current_time)
            self.next_tx_time = current_time + self.tx_interval + random.uniform(-10, 10)
            return success
        return False


class LoRaWAN_Network:
    """Rete LoRaWAN che gestisce tutti i nodi sensore."""

    def __init__(self, node_ids: List[str], tx_interval: int = 3600):
        self.gateway = Gateway()
        self.nodes: Dict[str, SensorNode] = {}
        self.current_time: float = 0.0

        print("\n📡 INIZIALIZZAZIONE RETE LoRaWAN...")
        for node_id in node_ids:
            sf = random.choice([7, 8, 9, 10, 11, 12])
            node = SensorNode(node_id, self.gateway, sf, tx_interval)
            self.nodes[node_id] = node
            print(f"   [+] Sensore registrato: {node_id} (SF: {sf})")

    def step(self, time_step: float) -> List[Packet]:
        """Esegue un passo di simulazione della comunicazione."""
        self.current_time += time_step
        for node in self.nodes.values():
            node.try_transmit(self.current_time)
        return self.gateway.get_and_clear_buffer()

    def update_node_tx_interval(self, node_id: str, new_interval: int):
        """Aggiorna l'intervallo di trasmissione di un nodo specifico."""
        if node_id in self.nodes:
            self.nodes[node_id].update_tx_interval(new_interval)
            print(f"   📡 {node_id}: intervallo aggiornato a {new_interval}s")

    def get_packet_loss_rate(self) -> float:
        """Restituisce la percentuale attuale di pacchetti persi."""
        return self.gateway.get_packet_loss_rate()

    def get_stats(self) -> dict:
        """Restituisce statistiche sulla rete."""
        return {
            'totale_inviati': self.gateway.stat_totale_inviati,
            'totale_ricevuti': self.gateway.stat_totale_ricevuti,
            'totale_persi': self.gateway.stat_totale_persi,
            'packet_loss_rate': self.get_packet_loss_rate()
        }

## 3. Configurazione Serbatoi IoT

In [ ]:
@dataclass
class TankConfig:
    """Configurazione per un serbatoio."""
    size_type: str
    tank_diameter: float
    pipe_diameter: float
    min_level: float = 0.0
    max_level: float = 12.0
    init_level: float = 10.0


TANK_CONFIGS = {
    'Small':  TankConfig('Small',  5.0,  0.15, 0.0, 8.0, 6.0),
    'Medium': TankConfig('Medium', 15.0, 0.30, 0.0, 12.0, 10.0),
    'Large':  TankConfig('Large',  40.0, 0.60, 0.0, 15.0, 12.0)
}

print("✅ Configurazioni serbatoi definite:")
for name, config in TANK_CONFIGS.items():
    print(f"   - {name}: diametro={config.tank_diameter}m, livello_max={config.max_level}m")

## 4. Classe per la Gestione della Rete Idrica

In [ ]:
class WaterNetworkManager:
    """Gestisce la rete idrica con serbatoi IoT."""

    def __init__(self, network_name: str = 'Net4'):
        self.network_name = network_name
        self.wn = None
        self.iot_tanks: Dict[str, dict] = {}
        self.valve_map: Dict[str, str] = {}
        
    def load_network(self):
        """Carica la rete Net3 (rete di test standard WNTR)."""
        print(f"\n💧 CARICAMENTO RETE IDRICA: {self.network_name}...")
        inp_file = '/usr/local/lib/python3.12/site-packages/wntr/library/networks/Net3.inp'
        self.wn = wntr.network.WaterNetworkModel(inp_file)
        print(f"   ✅ Rete caricata: {self.wn.num_nodes} nodi, {self.wn.num_links} collegamenti")
        print(f"   - Nodi: {self.wn.num_junctions} junction, {self.wn.num_tanks} tank, {self.wn.num_reservoirs} reservoir")
        return self
    
    def remove_existing_tanks(self):
        """Rimuove tutti i serbatoi esistenti dalla rete."""
        print("\n🗑️  RIMOZIONE SERBATOI ESISTENTI...")
        tanks_to_remove = list(self.wn.tank_names)
        for tank_name in tanks_to_remove:
            try:
                controls = [c for c in self.wn.controls() if hasattr(c, 'control_action') and getattr(c.control_action, 'target', None) == self.wn.get_node(tank_name)]
                for ctrl in controls:
                    self.wn.remove_control(ctrl)
                self.wn.remove_tank(tank_name)
                print(f"   [-] Rimosso serbatoio: {tank_name}")
            except Exception as e:
                print(f"   ⚠️  Errore rimozione {tank_name}: {e}")
        print(f"   ✅ Rimossi {len(tanks_to_remove)} serbatoi esistenti")
        return self
    
    def add_iot_tanks(self, num_tanks: int = 8):
        """Aggiunge serbatoi IoT in posizioni casuali."""
        print(f"\n➕ AGGIUNTA {num_tanks} SERBATOI IoT...")
        junctions = list(self.wn.junction_names)
        selected_junctions = random.sample(junctions, min(num_tanks, len(junctions)))
        tank_types = list(TANK_CONFIGS.keys())
        
        for i, junction_name in enumerate(selected_junctions):
            tank_type = tank_types[i % len(tank_types)]
            config = TANK_CONFIGS[tank_type]
            tank_name = f"IOT_TANK_{i+1:03d}"
            valve_name = f"IOT_VALVE_{i+1:03d}"
            
            junction = self.wn.get_node(junction_name)
            base_elevation = junction.elevation
            
            self.wn.add_tank(
                name=tank_name,
                elevation=base_elevation,
                init_level=config.init_level,
                min_level=config.min_level,
                max_level=config.max_level,
                diameter=config.tank_diameter,
                min_vol=0.0,
                vol_curve=None
            )
            
            self.wn.add_valve(
                name=valve_name,
                start_node_name=junction_name,
                end_node_name=tank_name,
                diameter=config.pipe_diameter,
                valve_type='PRV',
                minor_loss=0.0,
                initial_setting=0.0
            )
            
            self.iot_tanks[tank_name] = {
                'type': tank_type,
                'config': config,
                'valve': valve_name,
                'junction': junction_name,
                'sensor_id': f"SENSOR_{i+1:03d}"
            }
            self.valve_map[valve_name] = tank_name
            print(f"   [+] {tank_name} ({tank_type}) @ {junction_name} -> valvola: {valve_name}")
        
        print(f"   ✅ Aggiunti {len(self.iot_tanks)} serbatoi IoT")
        return self
    
    def open_tank_valve(self, tank_name: str):
        """Apre la valvola di un serbatoio per rilasciare acqua."""
        if tank_name not in self.iot_tanks:
            print(f"   ⚠️  Serbatoio {tank_name} non trovato")
            return False
        valve_name = self.iot_tanks[tank_name]['valve']
        valve = self.wn.get_link(valve_name)
        valve.initial_setting = 100.0
        print(f"   🔓 Valvola {valve_name} APERTA per serbatoio {tank_name}")
        return True
    
    def close_tank_valve(self, tank_name: str):
        """Chiude la valvola di un serbatoio."""
        if tank_name not in self.iot_tanks:
            print(f"   ⚠️  Serbatoio {tank_name} non trovato")
            return False
        valve_name = self.iot_tanks[tank_name]['valve']
        valve = self.wn.get_link(valve_name)
        valve.initial_setting = 0.0
        print(f"   🔒 Valvola {valve_name} CHIUSA per serbatoio {tank_name}")
        return True
    
    def get_tank_level(self, tank_name: str) -> float:
        """Ottiene il livello corrente di un serbatoio."""
        if tank_name not in self.wn.tank_names:
            return 0.0
        tank = self.wn.get_node(tank_name)
        return tank.level
    
    def get_demand_satisfaction(self) -> Tuple[float, float]:
        """Calcola quanto la domanda è soddisfatta."""
        total_demand = 0.0
        satisfied_demand = 0.0
        for junction_name in self.wn.junction_names:
            junction = self.wn.get_node(junction_name)
            demand = junction.demand_timeseries_list.at(0)
            total_demand += demand
            satisfied_demand += demand
        return total_demand, satisfied_demand
    
    def simulate_hydraulics(self, duration: int = 3600):
        """Esegue una simulazione idraulica semplificata."""
        for tank_name, info in self.iot_tanks.items():
            tank = self.wn.get_node(tank_name)
            valve = self.wn.get_link(info['valve'])
            
            if valve.initial_setting > 0:
                current_level = tank.level
                discharge_rate = 0.01 * valve.initial_setting
                tank_area = math.pi * (tank.diameter / 2) ** 2
                level_change = (discharge_rate * duration) / tank_area
                new_level = max(tank.min_level, current_level - level_change)
                tank.initial_level = new_level
            else:
                current_level = tank.level
                fill_rate = 0.001
                tank_area = math.pi * (tank.diameter / 2) ** 2
                level_change = (fill_rate * duration) / tank_area
                new_level = min(tank.max_level, current_level + level_change)
                tank.initial_level = new_level
        return self

## 5. Agente di Gestione Crisi Idrica

In [ ]:
class CrisisManagementAgent:
    """
    Agente che gestisce la crisi idrica.
    Funzione obiettivo:
    - Massimizza il guadagno in soddisfazione della domanda
    - Minimizza il tempo impiegato per realizzare l'azione
    - Minimizza la penalità per pacchetti persi
    """

    def __init__(self, water_network: WaterNetworkManager, lora_network: LoRaWAN_Network):
        self.wn_manager = water_network
        self.lora_network = lora_network
        self.crisis_active = False
        self.action_history: List[dict] = []
        self.alpha = 1.0
        self.beta = 0.1
        self.gamma = 0.3
        self.demand_threshold = 0.85
        self.packet_loss_threshold = 20.0
        
        print("\n🤖 AGENTE DI GESTIONE CRISI INITIALIZZATO")
        print(f"   - Soglia attivazione crisi: {self.demand_threshold*100}%")
        print(f"   - Soglia packet loss: {self.packet_loss_threshold}%")
    
    def compute_objective_function(self, satisfaction_before: float, satisfaction_after: float,
                                   time_elapsed: float, packet_loss_rate: float) -> float:
        """Calcola la funzione obiettivo."""
        satisfaction_gain = satisfaction_after - satisfaction_before
        time_penalty = time_elapsed / 3600.0
        loss_penalty = packet_loss_rate / 100.0
        objective = (self.alpha * satisfaction_gain) - (self.beta * time_penalty) - (self.gamma * loss_penalty)
        return objective
    
    def assess_situation(self) -> dict:
        """Valuta la situazione attuale della rete."""
        total_demand, satisfied_demand = self.wn_manager.get_demand_satisfaction()
        if total_demand == 0:
            satisfaction_ratio = 1.0
        else:
            satisfaction_ratio = satisfied_demand / total_demand
        packet_loss = self.lora_network.get_packet_loss_rate()
        return {
            'total_demand': total_demand,
            'satisfied_demand': satisfied_demand,
            'satisfaction_ratio': satisfaction_ratio,
            'packet_loss_rate': packet_loss,
            'crisis_active': self.crisis_active
        }
    
    def decide_action(self, situation: dict) -> List[str]:
        """Decide quali azioni intraprendere."""
        actions = []
        if situation['satisfaction_ratio'] < self.demand_threshold:
            self.crisis_active = True
            print(f"\n🚨 CRISI RILEVATA! Soddisfazione: {situation['satisfaction_ratio']*100:.1f}%")
            sorted_tanks = sorted(
                self.wn_manager.iot_tanks.items(),
                key=lambda x: TANK_CONFIGS[x[1]['type']].tank_diameter,
                reverse=True
            )
            for tank_name, info in sorted_tanks:
                tank_level = self.wn_manager.get_tank_level(tank_name)
                tank = self.wn_manager.wn.get_node(tank_name)
                if tank_level > tank.min_level + 1.0:
                    actions.append(tank_name)
                    print(f"   → Decisione: APRIRE {tank_name} (livello: {tank_level:.2f}m)")
        else:
            self.crisis_active = False
            print(f"\n✅ Situazione normale. Soddisfazione: {situation['satisfaction_ratio']*100:.1f}%")
        return actions
    
    def adjust_sensor_frequency(self, situation: dict):
        """Regola dinamicamente la frequenza di invio dei sensori."""
        packet_loss = situation['packet_loss_rate']
        if packet_loss > self.packet_loss_threshold:
            new_interval = 7200
            print(f"   📉 Packet loss alto ({packet_loss:.1f}%): riduco frequenza a {new_interval}s")
        elif self.crisis_active:
            new_interval = 300
            print(f"   📈 Crisi attiva: aumento frequenza a {new_interval}s")
        else:
            new_interval = 3600
            print(f"   📊 Situazione normale: frequenza standard {new_interval}s")
        for sensor_id in self.wn_manager.iot_tanks.values():
            self.lora_network.update_node_tx_interval(sensor_id['sensor_id'], new_interval)
    
    def execute_action(self, tank_names: List[str]) -> dict:
        """Esegue le azioni decise."""
        if not tank_names:
            return {'tanks_opened': 0, 'success': True}
        _, satisfied_before = self.wn_manager.get_demand_satisfaction()
        total_demand, _ = self.wn_manager.get_demand_satisfaction()
        satisfaction_before = satisfied_before / total_demand if total_demand > 0 else 1.0
        opened_count = 0
        for tank_name in tank_names:
            if self.wn_manager.open_tank_valve(tank_name):
                opened_count += 1
        time_elapsed = opened_count * 60
        _, satisfied_after = self.wn_manager.get_demand_satisfaction()
        satisfaction_after = satisfied_after / total_demand if total_demand > 0 else 1.0
        packet_loss = self.lora_network.get_packet_loss_rate()
        objective_value = self.compute_objective_function(
            satisfaction_before, satisfaction_after, time_elapsed, packet_loss
        )
        action_record = {
            'tanks_opened': tank_names,
            'satisfaction_before': satisfaction_before,
            'satisfaction_after': satisfaction_after,
            'time_elapsed': time_elapsed,
            'packet_loss': packet_loss,
            'objective_value': objective_value
        }
        self.action_history.append(action_record)
        print(f"\n📊 RISULTATO AZIONE:")
        print(f"   - Serbatoi aperti: {opened_count}")
        print(f"   - Soddisfazione prima: {satisfaction_before*100:.1f}%")
        print(f"   - Soddisfazione dopo: {satisfaction_after*100:.1f}%")
        print(f"   - Tempo impiegato: {time_elapsed}s")
        print(f"   - Packet loss: {packet_loss:.1f}%")
        print(f"   - Valore funzione obiettivo: {objective_value:.4f}")
        return action_record
    
    def run_decision_cycle(self) -> dict:
        """Esegue un ciclo completo di decisione."""
        print("\n" + "="*60)
        print("🔄 CICLO DECISIONALE DELL'AGENTE")
        print("="*60)
        situation = self.assess_situation()
        print(f"\n📋 VALUTAZIONE SITUAZIONE:")
        print(f"   - Domanda totale: {situation['total_demand']:.2f} m³/s")
        print(f"   - Domanda soddisfatta: {situation['satisfied_demand']:.2f} m³/s")
        print(f"   - Rapporto soddisfazione: {situation['satisfaction_ratio']*100:.1f}%")
        print(f"   - Packet loss: {situation['packet_loss_rate']:.1f}%")
        actions = self.decide_action(situation)
        result = self.execute_action(actions)
        self.adjust_sensor_frequency(situation)
        return result

## 6. Co-Simulazione Integrata

In [ ]:
class CoSimulation:
    """Gestisce la co-simulazione tra dominio idrico e dominio sensoristico."""

    def __init__(self):
        self.water_network = None
        self.lora_network = None
        self.agent = None
        self.statistics: List[dict] = []
        self.current_time = 0
        
    def setup(self, num_tanks: int = 8):
        """Configura la co-simulazione."""
        print("\n" + "="*60)
        print("🚀 SETUP CO-SIMULAZIONE")
        print("="*60)
        self.water_network = WaterNetworkManager('Net4')
        self.water_network.load_network()
        self.water_network.remove_existing_tanks()
        self.water_network.add_iot_tanks(num_tanks)
        sensor_ids = [info['sensor_id'] for info in self.water_network.iot_tanks.values()]
        self.lora_network = LoRaWAN_Network(sensor_ids, tx_interval=3600)
        self.agent = CrisisManagementAgent(self.water_network, self.lora_network)
        print("\n✅ SETUP COMPLETATO")
        return self
    
    def step(self, time_step: int = 3600):
        """Esegue un passo di co-simulazione."""
        print(f"\n⏱️  PASSO {self.current_time // 3600}h - {time_step}s")
        print("-" * 40)
        for tank_name, info in self.water_network.iot_tanks.items():
            pressure = self.water_network.get_tank_level(tank_name)
            sensor_id = info['sensor_id']
            if sensor_id in self.lora_network.nodes:
                self.lora_network.nodes[sensor_id].update_sensor_data(pressure=pressure, tank_level=pressure)
        packets = self.lora_network.step(time_step)
        if packets:
            print(f"   📡 {len(packets)} pacchetti ricevuti dal gateway")
        decision_result = self.agent.run_decision_cycle()
        self.water_network.simulate_hydraulics(time_step)
        stats = {
            'time': self.current_time,
            'packets_received': len(packets),
            'packet_loss_rate': self.lora_network.get_packet_loss_rate(),
            'crisis_active': self.agent.crisis_active,
            'tanks_opened': len(decision_result.get('tanks_opened', [])),
            'satisfaction_before': decision_result.get('satisfaction_before', 0),
            'satisfaction_after': decision_result.get('satisfaction_after', 0),
            'objective_value': decision_result.get('objective_value', 0)
        }
        self.statistics.append(stats)
        self.current_time += time_step
        return stats
    
    def run(self, num_steps: int = 24, time_step: int = 3600):
        """Esegue la co-simulazione."""
        print("\n" + "="*60)
        print(f"▶️  AVVIO CO-SIMULAZIONE: {num_steps} passi da {time_step}s")
        print(f"   Tempo totale simulato: {num_steps * time_step / 3600} ore")
        print("="*60)
        for i in range(num_steps):
            self.step(time_step)
        print("\n" + "="*60)
        print("✅ CO-SIMULAZIONE COMPLETATA")
        print("="*60)
        return self.statistics
    
    def print_summary(self):
        """Stampa un riassunto della simulazione."""
        if not self.statistics:
            print("Nessuna statistica disponibile")
            return
        print("\n" + "="*60)
        print("📊 RIEPILOGO CO-SIMULAZIONE")
        print("="*60)
        total_packets = sum(s['packets_received'] for s in self.statistics)
        avg_packet_loss = sum(s['packet_loss_rate'] for s in self.statistics) / len(self.statistics)
        crisis_steps = sum(1 for s in self.statistics if s['crisis_active'])
        total_tanks_opened = sum(s['tanks_opened'] for s in self.statistics)
        avg_objective = sum(s['objective_value'] for s in self.statistics) / len(self.statistics)
        print(f"\n📈 METRICHE PRESTAZIONALI:")
        print(f"   - Pacchetti totali ricevuti: {total_packets}")
        print(f"   - Packet loss medio: {avg_packet_loss:.2f}%")
        print(f"   - Passi con crisi attiva: {crisis_steps}/{len(self.statistics)}")
        print(f"   - Totale aperture serbatoi: {total_tanks_opened}")
        print(f"   - Valore medio funzione obiettivo: {avg_objective:.4f}")
        lora_stats = self.lora_network.get_stats()
        print(f"\n📡 STATISTICHE LoRaWAN:")
        print(f"   - Pacchetti inviati: {lora_stats['totale_inviati']}")
        print(f"   - Pacchetti ricevuti: {lora_stats['totale_ricevuti']}")
        print(f"   - Pacchetti persi: {lora_stats['totale_persi']}")
        print(f"   - Packet loss rate: {lora_stats['packet_loss_rate']:.2f}%")
        print(f"\n🤖 AZIONI DELL'AGENTE:")
        print(f"   - Cicli decisionali eseguiti: {len(self.agent.action_history)}")
        if self.agent.action_history:
            best_action = max(self.agent.action_history, key=lambda x: x['objective_value'])
            print(f"   - Miglior azione (obiettivo: {best_action['objective_value']:.4f}):")
            print(f"     • Serbatoi: {best_action['tanks_opened']}")
            print(f"     • Soddisfazione: {best_action['satisfaction_before']*100:.1f}% → {best_action['satisfaction_after']*100:.1f}%")

## 7. Esecuzione della Co-Simulazione

In [ ]:
random.seed(42)
co_sim = CoSimulation()
co_sim.setup(num_tanks=8)

In [ ]:
statistics = co_sim.run(num_steps=24, time_step=3600)

In [ ]:
co_sim.print_summary()

## 8. Visualizzazione Risultati

In [ ]:
import matplotlib.pyplot as plt

times = [s['time'] / 3600 for s in statistics]
packet_loss = [s['packet_loss_rate'] for s in statistics]
satisfaction = [s['satisfaction_after'] for s in statistics]
objective_values = [s['objective_value'] for s in statistics]
crisis_flags = [1 if s['crisis_active'] else 0 for s in statistics]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Risultati Co-Simulazione Gestione Crisi Idrica', fontsize=16, fontweight='bold')

axes[0, 0].plot(times, packet_loss, 'b-', linewidth=2, marker='o')
axes[0, 0].axhline(y=20, color='r', linestyle='--', label='Soglia critica (20%)')
axes[0, 0].set_xlabel('Tempo (ore)')
axes[0, 0].set_ylabel('Packet Loss (%)')
axes[0, 0].set_title('Packet Loss nella Rete LoRaWAN')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(times, satisfaction, 'g-', linewidth=2, marker='s')
axes[0, 1].axhline(y=0.85, color='r', linestyle='--', label='Soglia crisi (85%)')
axes[0, 1].set_xlabel('Tempo (ore)')
axes[0, 1].set_ylabel('Rapporto di Soddisfazione')
axes[0, 1].set_title('Soddisfazione della Domanda Idrica')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(times, objective_values, 'purple', linewidth=2, marker='d')
axes[1, 0].axhline(y=0, color='gray', linestyle='-', alpha=0.5)
axes[1, 0].set_xlabel('Tempo (ore)')
axes[1, 0].set_ylabel('Valore Funzione Obiettivo')
axes[1, 0].set_title('Andamento Funzione Obiettivo dell\'Agente')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].bar(times, crisis_flags, color='red', alpha=0.6, label='Crisi Attiva')
axes[1, 1].set_xlabel('Tempo (ore)')
axes[1, 1].set_ylabel('Stato')
axes[1, 1].set_title('Stati di Crisi Rilevati')
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_yticklabels(['Normale', 'Crisi'])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n✅ Visualizzazione completata!")

## Conclusione

Questo notebook implementa un sistema di co-simulazione che integra:

1. **Dominio Idrico** (WNTR):
   - Rete Net4 modificata con serbatoi IoT
   - Tre tipologie di serbatoi (Small, Medium, Large)
   - Valvole controllabili per gestione flusso

2. **Dominio Sensoristico** (LoRaWAN simulato):
   - Sensori associati a ogni serbatoio
   - Gateway con rilevamento collisioni
   - Calcolo packet loss

3. **Agente di Gestione Crisi**:
   - Funzione obiettivo multi-parametro
   - Decisioni basate su soddisfazione della domanda
   - Regolazione dinamica frequenza sensori

Il sistema dimostra come l'integrazione tra dominio fisico (idrico) e cyber (sensori) possa ottimizzare la gestione delle risorse idriche durante situazioni di crisi.